# Physical Activity

In [2]:
%pip install -q -r ../../requirements.txt

# If this block is stuck: 
# Press: Ctrl + Shift + P
# Run: Developer: Reload Window

Note: you may need to restart the kernel to use updated packages.


In [3]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# from scipy.stats import entropy
import pandas as pd
import os
# import numpy as np

In [ ]:
# Load CSV files
data_dir = '../../mcphases/'

active_minutes = pd.read_csv(os.path.join(data_dir, 'active_minutes.csv'))
calories = pd.read_csv(os.path.join(data_dir, 'calories.csv'))
# demographic_vo2_max = pd.read_csv(os.path.join(data_dir, 'demographic_vo2_max.csv'))
# exercise = pd.read_csv(os.path.join(data_dir, 'exercise.csv'))
time_in_heart_rate_zones = pd.read_csv(os.path.join(data_dir, 'time_in_heart_rate_zones.csv'))
# height_and_weight = pd.read_csv(os.path.join(data_dir, 'height_and_weight.csv'))
subject_info = pd.read_csv(os.path.join(data_dir, 'subject-info.csv'))
hormones_and_selfreport = pd.read_csv(os.path.join(data_dir, 'hormones_and_selfreport.csv'))

print("All CSV files loaded successfully!")

All CSV files loaded successfully!


### Prepare the individual dataframes

To include from every dataframe: id, day_in_study, is_weekend

active_minutes: sedentary, lightly, moderately, very

calories: calories (daily sum based on day_in_study)

(ignore) demographic_vo2_max: filtered_demographic_vo2_max

(ignore) exercise: start_day_in_study as day_in_study, 
    activitylevel (looks like: "[{'minutes': 0, 'name': 'sedentary'}, {'minutes': 3, 'name': 'lightly'}, {'minutes': 11, 'name': 'fairly'}, {'minutes': 2, 'name': 'very'}]"), 
    heartratezones (looks like: "[{'name': 'Out of Range', 'min': 30, 'max': 121, 'minutes': 16, 'caloriesOut': 69.75168}, {'name': 'Fat Burn', 'min': 121, 'max': 143, 'minutes': 0, 'caloriesOut': 0.0}, {'name': 'Cardio', 'min': 143, 'max': 171, 'minutes': 0, 'caloriesOut': 0.0}, {'name': 'Peak', 'min': 171, 'max': 220, 'minutes': 0, 'caloriesOut': 0.0}]")

time_in_heart_rate_zones: in_default_zone_3, in_default_zone_2, in_default_zone_1, below_default_zone_1

(ignore) height_and_weight: height_2022, height_2024, weight_2022, weight_2024 --> calculate BMI

subject_info: birth_year --> age, ethnicity, sexually_active, self_report_menstrual_health_literacy --> convert to numerical feature, age_of_first_menarche

hormones_and_selfreport: phase, lh, estrogen, pdg, exerciselevel, fatigue

#### Active minutes

In [10]:
active_minutes = active_minutes.drop(columns=["study_interval"])
active_minutes.head()

,id,is_weekend,day_in_study,sedentary,lightly,moderately,very
0,1,True,1,753.0,64,0,0
1,1,False,2,855.0,74,0,0
2,1,False,3,751.0,134,18,7
3,1,False,4,905.0,86,0,0
4,1,False,5,1430.0,10,0,0


#### Calories

In [11]:
daily_calories = (
    calories
    .groupby(["id", "day_in_study"], as_index=False)
    .agg(
        is_weekend=("is_weekend", "first"),
        calories_sum=("calories", "sum")
    )
)

daily_calories.head()

,id,day_in_study,is_weekend,calories_sum
0,1,1,True,1542.0
1,1,2,False,1591.0
2,1,3,False,1755.0
3,1,4,False,1552.0
4,1,5,False,1456.0


#### Demographic VO2 max (ignore)

In [45]:
demographic_vo2_max["filtered_demographic_vo2_max"].describe()

count    11482.000000
mean      7822.533262
std      10199.147630
min         24.548290
25%         34.920210
50%         43.321700
75%      21474.836470
max      21474.836470
Name: filtered_demographic_vo2_max, dtype: float64

The normal distribution of VO2 is below 10-80. Something certainly went wrong.

In [46]:
print(demographic_vo2_max[
    demographic_vo2_max["filtered_demographic_vo2_max"] > 100
].shape)

print(demographic_vo2_max.shape)

(4763, 8)
(11482, 8)


About 40% of the data is invalid -- do not make sense.

In [47]:
(
    demographic_vo2_max.loc[
        demographic_vo2_max["filtered_demographic_vo2_max"] > 100,
        "filtered_demographic_vo2_max"
    ]
    .value_counts()
    .head(20)
)

filtered_demographic_vo2_max
21474.83647    4071
1719.30929       30
17846.72171      16
12836.16711      16
15055.24327      16
9530.59761       16
8271.08867       16
7201.50779       16
6286.00887       16
5497.53187       16
4815.24668       16
4222.81563       16
11029.01284      16
3706.93104       16
3256.65093       16
2518.26935       16
2862.93615       16
2216.34569       16
1515.41435       16
1336.48290       16
Name: count, dtype: int64

In [48]:
demographic_vo2_max.groupby("id")[
    "filtered_demographic_vo2_max"
].max().sort_values(ascending=False).head(20)

id
9     21474.83647
10    21474.83647
22    21474.83647
26    21474.83647
30    21474.83647
20    21474.83647
18    21474.83647
14    21474.83647
13    21474.83647
12    21474.83647
50    21474.83647
43    21474.83647
48    21474.83647
47    21474.83647
42    21474.83647
41    21474.83647
38    21474.83647
33    21474.83647
32    21474.83647
27    21474.83647
Name: filtered_demographic_vo2_max, dtype: float64

Invalid data is not from a single mal-functioning device, but rather across participants.

In [56]:
demographic_vo2_max.groupby("study_interval")[
    "filtered_demographic_vo2_max"
].describe()

,count,mean,std,min,25%,50%,75%,max
study_interval,,,,,,,,
2022,3531.0,38.719237,4.977277,26.89121,35.233445,39.02958,41.789405,52.51878
2024,7951.0,11279.286792,10552.894565,24.54829,34.197280,21474.83647,21474.836470,21474.83647


In [57]:
demographic_vo2_max.groupby("study_interval").apply(
    lambda x: (
        (x["filtered_demographic_vo2_max"] > 100) |
        (x["filtered_demographic_vo2_max"] < 10)
    ).mean() * 100
)

C:\Users\caowe\AppData\Local\Temp\ipykernel_31796\325272722.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  demographic_vo2_max.groupby("study_interval").apply(


study_interval
2022     0.000000
2024    59.904415
dtype: float64

It seems that all corrupted data are from the second study interval.

In [49]:
# Remove outliers from demographic_vo2_max
demographic_vo2_max = demographic_vo2_max[
    demographic_vo2_max["filtered_demographic_vo2_max"] <= 100]

demographic_vo2_max["filtered_demographic_vo2_max"].describe()

count    6719.000000
mean       36.318996
std         9.352231
min        24.548290
25%        26.599060
50%        36.225680
75%        41.246375
max        93.726240
Name: filtered_demographic_vo2_max, dtype: float64

In [51]:
demographic_vo2_max = demographic_vo2_max.drop(columns=["study_interval", "demographic_vo2_max", 
                                                        "demographic_vo2_max_error"])
demographic_vo2_max.head()

,id,is_weekend,day_in_study,filtered_demographic_vo2_max,filtered_demographic_vo2_max_error
0,1,True,1,33.79370,3.00000
1,1,False,2,32.55987,1.51239
2,1,False,3,31.50628,1.02734
3,1,False,4,31.06774,0.79267
4,1,False,5,30.88130,0.65787


In [27]:
demographic_vo2_daily = (
    demographic_vo2_max
    .groupby(["id", "day_in_study", "is_weekend"], as_index=False)
    .agg({
        "filtered_demographic_vo2_max": "mean"
    })
)

#### Exercise (ignore)

In [21]:
print("Total rows:", len(exercise))
print("Exact duplicates:", exercise.duplicated().sum())

Total rows: 7282
Exact duplicates: 3641


In [22]:
exercise.groupby(list(exercise.columns)).size().value_counts()

1    2382
5     363
6     242
2     111
7      45
8      30
4       1
Name: count, dtype: int64

In [44]:
exercise_unique = exercise.drop_duplicates()
print("Exact duplicates for new dataframe:", exercise_unique.duplicated().sum())

Exact duplicates for new dataframe: 0


In [49]:
exercise_unique = exercise_unique.drop(columns=["study_interval", "last_modified_day_in_study", "last_modified_timestamp", "activitytypeid",
                                                "original_start_day_in_study", "original_start_timestamp", "originalduration", "duration",
                                                "manualvaluesspecified", "logtype", "hasgps", "shouldfetchdetails", "hasactivezoneminutes"])
exercise_unique = exercise_unique.rename(columns={"start_day_in_study": "day_in_study"})
exercise_unique.head(5)

,id,is_weekend,day_in_study,start_timestamp,activityname,activitylevel,averageheartrate,calories,activeduration,steps,heartratezones,activezoneminutes,elevationgain
0,1,False,39,21:23:14,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",106.0,96,1127000,1848.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 1, 'minutesInHeartRateZones':...",18.288
1,1,False,39,21:23:14,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",106.0,96,1127000,1848.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 1, 'minutesInHeartRateZones':...",18.288
7,1,False,44,19:56:44,Outdoor Bike,"[{'minutes': 24, 'name': 'sedentary'}, {'minut...",91.0,30,1639000,NaN,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",0.000
10,1,False,44,18:33:58,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",100.0,268,3430000,5356.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",3.454
14,1,False,44,18:33:58,Walk,"[{'minutes': 0, 'name': 'sedentary'}, {'minute...",100.0,268,3430000,5356.0,"[{'name': 'Out of Range', 'min': 30, 'max': 12...","{'totalMinutes': 0, 'minutesInHeartRateZones':...",3.454


#### Time in heart rate zones

In [13]:
time_in_heart_rate_zones = time_in_heart_rate_zones.drop(columns=["study_interval"])
time_in_heart_rate_zones.head()

,id,is_weekend,day_in_study,in_default_zone_3,in_default_zone_2,in_default_zone_1,below_default_zone_1
0,1,True,1,0.0,0.0,126.0,1036.0
1,1,False,2,5.0,82.0,416.0,512.0
2,1,False,3,5.0,119.0,599.0,368.0
3,1,False,4,0.0,0.0,212.0,613.0
4,1,False,5,8.0,123.0,250.0,308.0


In [14]:
time_in_heart_rate_zones = time_in_heart_rate_zones.rename(
    columns={"in_default_zone_3": "peak_zone", 
             "in_default_zone_2": "cardio_zone", 
             "in_default_zone_1": "fat_burn_zone", 
             "below_default_zone_1": "below_fat_burn_zone"})
time_in_heart_rate_zones.head()

,id,is_weekend,day_in_study,peak_zone,cardio_zone,fat_burn_zone,below_fat_burn_zone
0,1,True,1,0.0,0.0,126.0,1036.0
1,1,False,2,5.0,82.0,416.0,512.0
2,1,False,3,5.0,119.0,599.0,368.0
3,1,False,4,0.0,0.0,212.0,613.0
4,1,False,5,8.0,123.0,250.0,308.0


In [15]:
time_in_heart_rate_zones["id"].unique()

array([ 1,  2,  3,  4,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 18, 19,
       20, 22, 23, 24, 26, 27, 29, 30, 32, 33, 34, 37, 38, 39, 40, 41, 42,
       43, 44, 45, 46, 47, 48, 49, 50])

#### Height and Weight (ignore)

In [65]:
height_and_weight.head()

,id,height_2022,weight_2022,height_2024,weight_2024
0,1,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN
2,3,167.0,52.0,NaN,NaN
3,4,170.0,78.0,NaN,NaN
4,6,NaN,NaN,NaN,NaN


In [ ]:
height_and_weight["height"] = height_and_weight[["height_2022", "height_2024"]].max(axis=1)
height_and_weight = height_and_weight.drop(columns=["height_2022", "height_2024"])
height_and_weight.head(100)

,id,weight_2022,weight_2024,height
0,1,NaN,NaN,NaN
1,2,NaN,NaN,NaN
2,3,52.0,NaN,167.0
3,4,78.0,NaN,170.0
4,6,NaN,NaN,NaN
5,7,56.0,NaN,170.0
6,8,NaN,NaN,NaN
7,9,50.8,53.5,160.0
8,10,60.3,NaN,161.0
9,11,NaN,NaN,NaN


In [98]:
height_and_weight["weight"] = (
    height_and_weight["weight_2024"]
    .fillna(height_and_weight["weight_2022"])
)

height_and_weight["BMI"] = (
    height_and_weight["weight"] /
    (height_and_weight["height"] / 100) ** 2
)

height_and_weight.head(50)

,id,weight_2022,weight_2024,height,BMI,weight
0,1,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN
2,3,52.0,NaN,167.0,18.645344,52.0
3,4,78.0,NaN,170.0,26.989619,78.0
4,6,NaN,NaN,NaN,NaN,NaN
5,7,56.0,NaN,170.0,19.377163,56.0
6,8,NaN,NaN,NaN,NaN,NaN
7,9,50.8,53.5,160.0,20.898437,53.5
8,10,60.3,NaN,161.0,23.262991,60.3
9,11,NaN,NaN,NaN,NaN,NaN


In [103]:
print("Out of 42 participants,\n", height_and_weight["height"].isna().sum(), "participants have missing height data,\n", 
      height_and_weight["weight_2022"].isna().sum(), "participants have missing weight data in 2022,\n", 
      height_and_weight["weight_2024"].isna().sum(), "participants have missing weight data in 2024.")
total_missing = height_and_weight["height"].isna().sum() + height_and_weight["weight_2022"].isna().sum() + height_and_weight["weight_2024"].isna().sum()
print("That is a total of", total_missing, "missing values in the height and weight dataset.",
      "which is", (total_missing / (42 * 3) * 100), "% of the total data.")

Out of 42 participants,
 17 participants have missing height data,
 18 participants have missing weight data in 2022,
 31 participants have missing weight data in 2024.
That is a total of 66 missing values in the height and weight dataset. which is 52.38095238095239 % of the total data.


In [106]:
print("There are", height_and_weight["BMI"].isna().sum(), "missing BMI values, out of 42 participants," \
"which is", (height_and_weight["BMI"].isna().sum() / 42 * 100), "% of the total data.")

There are 18 missing BMI values, out of 42 participants,which is 42.857142857142854 % of the total data.


#### Subject info

In [16]:
# subject_info: birth_year --> age
subject_info["age"] = 2024 - subject_info["birth_year"]
subject_info = subject_info.drop(columns=["birth_year", "gender", "education", "ethnicity"])
subject_info.head()

,id,sexually_active,self_report_menstrual_health_literacy,age_of_first_menarche,age
0,1,Yes,NaN,14,25
1,2,Yes,High,13,29
2,3,No,High,12,24
3,4,No,Medium,12,24
4,6,Yes,Low,13,27


In [17]:
# Convert self-report menstrual health literacy to numeric values
literacy_mapping = {
    "Non-existent": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Expert": 4
}

subject_info["menstrual_health_literacy_num"] = (
    subject_info["self_report_menstrual_health_literacy"]
    .map(literacy_mapping)
)
subject_info = subject_info.drop(columns=["self_report_menstrual_health_literacy"])
subject_info.head(100)

,id,sexually_active,age_of_first_menarche,age,menstrual_health_literacy_num
0,1,Yes,14,25,NaN
1,2,Yes,13,29,3.0
2,3,No,12,24,3.0
3,4,No,12,24,2.0
4,6,Yes,13,27,1.0
5,7,Yes,10,23,2.0
6,8,Yes,10,24,3.0
7,9,No,13,20,2.0
8,10,No,11,22,3.0
9,11,No,12,21,2.0


In [13]:
# subject_info["ethnicity"].value_counts()

In [14]:
# ethnicity_dummies = pd.get_dummies(
#     subject_info["ethnicity"],
#     prefix="ethnicity",
#     dtype=int
# )

# subject_info = pd.concat(
#     [subject_info, ethnicity_dummies],
#     axis=1
# )
# subject_info.head(100)

In [18]:
# Convert self-report sexually_active to numeric values
sexually_active_mapping = {
    "No": 0,
    "Yes": 1
}

subject_info["sexually_active_num"] = (
    subject_info["sexually_active"]
    .map(sexually_active_mapping)
)
subject_info = subject_info.drop(columns=["sexually_active"])
subject_info.head(100)

,id,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num
0,1,14,25,NaN,1.0
1,2,13,29,3.0,1.0
2,3,12,24,3.0,0.0
3,4,12,24,2.0,0.0
4,6,13,27,1.0,1.0
5,7,10,23,2.0,1.0
6,8,10,24,3.0,1.0
7,9,13,20,2.0,0.0
8,10,11,22,3.0,0.0
9,11,12,21,2.0,0.0


#### Hormones and self-report

In [19]:
# Convert self-report symptoms to numeric values
likert_map = {
    'Not at all': 0,
    'Very Low/Little': 1,
    'Low': 2,
    'Moderate': 3,
    'High': 4,
    'Very High': 5
}

symptoms = [
    'appetite',
    'exerciselevel',
    'headaches',
    'cramps',
    'sorebreasts',
    'fatigue',
    'sleepissue',
    'moodswing',
    'stress',
    'foodcravings',
    'indigestion',
    'bloating'
]

for col in symptoms:
    hormones_and_selfreport[col + '_num'] = (
        hormones_and_selfreport[col]
        .map(likert_map)
    )

In [20]:
hormones_and_selfreport.head()

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,headaches_num,cramps_num,sorebreasts_num,fatigue_num,sleepissue_num,moodswing_num,stress_num,foodcravings_num,indigestion_num,bloating_num
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,4.0,1.0,1.0,4.0,2.0,1.0,3.0,1.0,1.0,1.0
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,5.0,1.0,1.0,4.0,5.0,1.0,3.0,1.0,1.0,1.0
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,4.0,1.0,1.0,5.0,5.0,1.0,2.0,1.0,1.0,1.0
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,1.0,1.0,1.0,4.0,5.0,1.0,2.0,1.0,1.0,1.0
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,1.0,1.0,1.0,4.0,4.0,1.0,2.0,1.0,1.0,1.0


In [21]:
hormones_and_selfreport = hormones_and_selfreport[["id", "is_weekend", "day_in_study", "phase", "lh", 
                                                   "estrogen", "pdg", "exerciselevel_num", "fatigue_num"]]
hormones_and_selfreport.head(100)

,id,is_weekend,day_in_study,phase,lh,estrogen,pdg,exerciselevel_num,fatigue_num
0,1,True,1,Follicular,2.9,94.2,NaN,2.0,4.0
1,1,False,2,Follicular,1.2,226.3,NaN,2.0,4.0
2,1,False,3,Follicular,3.5,276.8,NaN,NaN,5.0
3,1,False,4,Fertility,1.8,322.1,NaN,2.0,4.0
4,1,False,5,Fertility,4.6,244.9,NaN,NaN,4.0
...,...,...,...,...,...,...,...,...,...
95,2,False,6,Fertility,4.6,117.3,NaN,2.0,3.0
96,2,True,7,Luteal,4.4,99.8,NaN,4.0,3.0
97,2,True,8,Luteal,8.1,241.2,NaN,2.0,4.0
98,2,False,9,Luteal,4.8,159.9,NaN,2.0,3.0


In [ ]:
print("The number of missing values in 'pdg':", hormones_and_selfreport["pdg"].isna().sum())
print("The number of non-missing values in 'pdg':", hormones_and_selfreport["pdg"].notna().sum())

The number of missing values in 'pdg': 3795
The number of non-missing values in 'pdg': 1864


In [24]:
1864/(1864+3795)

0.329386817458915

### Merge all dataframes into one dataframe

First merge:
- active_minutes: id, is_weekend, day_in_study;
- daily_calories: id, is_weekend, day_in_study;
- demographic_vo2_max: id, is_weekend, day_in_study;
- time_in_heart_rate_zones: id, is_weekend, day_in_study;
- hormones_and_selfreport: id, is_weekend, day_in_study;

Then merge:
- subject_info: id


In [ ]:
for name, df in {
    "active_minutes": active_minutes,
    "daily_calories": daily_calories,
    "time_in_heart_rate_zones": time_in_heart_rate_zones,
    "hormones_and_selfreport": hormones_and_selfreport,
}.items():

    dup = df.duplicated(
        subset=["id", "day_in_study", "is_weekend"]
    ).sum()

    print(f"{name}: {dup}")

active_minutes: 71
daily_calories: 0
demographic_vo2_max: 5990
time_in_heart_rate_zones: 103
hormones_and_selfreport: 0


It seems demographic_vo2_max has a lot of duplicated values...process this dataframe again.

In [ ]:
for name, df in {
    "active_minutes": active_minutes,
    "daily_calories": daily_calories,
    "time_in_heart_rate_zones": time_in_heart_rate_zones,
    "hormones_and_selfreport": hormones_and_selfreport,
}.items():

    print(
        name,
        df.groupby(
            ["id", "day_in_study", "is_weekend"]
        ).size().max()
    )

active_minutes 5
daily_calories 1
demographic_vo2_max 15
time_in_heart_rate_zones 5
hormones_and_selfreport 1


In [ ]:
# daily-level data: merge on id + day_in_study + is_weekend
merged_df = (
    active_minutes
    .merge(
        daily_calories,
        on=["id", "day_in_study", "is_weekend"],
        how="outer"
    )
    .merge(
        time_in_heart_rate_zones,
        on=["id", "day_in_study", "is_weekend"],
        how="outer"
    )
    .merge(
        hormones_and_selfreport,
        on=["id", "day_in_study", "is_weekend"],
        how="outer"
    )
)

# participant-level data: merge on id only
merged_df = merged_df.merge(
    subject_info,
    on="id",
    how="left"
)

print(merged_df.shape)
merged_df.head()

(14557, 23)


,id,is_weekend,day_in_study,sedentary,lightly,moderately,very,calories_sum,filtered_demographic_vo2_max,peak_zone,...,phase,lh,estrogen,pdg,exerciselevel_num,fatigue_num,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num
0,1,True,1,753.0,64.0,0.0,0.0,1542.0,33.79370,0.0,...,Follicular,2.9,94.2,NaN,2.0,4.0,14,25,NaN,1.0
1,1,False,2,855.0,74.0,0.0,0.0,1591.0,32.55987,5.0,...,Follicular,1.2,226.3,NaN,2.0,4.0,14,25,NaN,1.0
2,1,False,3,751.0,134.0,18.0,7.0,1755.0,31.50628,5.0,...,Follicular,3.5,276.8,NaN,NaN,5.0,14,25,NaN,1.0
3,1,False,4,905.0,86.0,0.0,0.0,1552.0,31.06774,0.0,...,Fertility,1.8,322.1,NaN,2.0,4.0,14,25,NaN,1.0
4,1,False,5,1430.0,10.0,0.0,0.0,1456.0,30.88130,8.0,...,Fertility,4.6,244.9,NaN,NaN,4.0,14,25,NaN,1.0


In [21]:
merged_df.columns

Index(['id', 'is_weekend', 'day_in_study', 'sedentary', 'lightly',
       'moderately', 'very', 'calories_sum', 'filtered_demographic_vo2_max',
       'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone',
       'phase', 'lh', 'estrogen', 'pdg', 'exerciselevel_num', 'fatigue_num',
       'age_of_first_menarche', 'age', 'menstrual_health_literacy_num',
       'sexually_active_num'],
      dtype='object')

In [22]:
merged_df.to_csv("../../mcphases/merged/physical_activity_merged.csv", index=False)